# 2.6 Pipelines

## Problema

Tenemos nuestro modelo en producción. Nuestro API de Flask carga nuestro modelo, y tenemos un endpoint para enviar datos y realizar alguna predicción con éstos:

```bash
curl -X POST http://127.0.0.1:5000/predecir -d '{"input": [0,1,0.6159084,0,0,0.55547282,1]}'
```

Lamentablemente, nuestro API no es el más práctico del mundo. El formato en el que pide los datos es muy poco amigable.

Podría ser algo así:

```json
{
  "Pclass": 0,
  "Sex": 1,
  "Age": 0.6159084,
  "SibSp": 0,
  "Parch": 0,
  "Fare": 0.55547282,
  "Embarked": 1
}
```

Esto estaría mucho mejor porque mínimo sabríamos los valores que estamos asignando a cada feature. No obstante, **¿qué significa una edad de 0.6159084?**

Recordemos que en nuestro proceso de feature engineering realizamos transformaciones significativas: conversión a variables numéricas, normalización y escalamiento. No podemos esperar que los usuarios de nuestro API realicen estas transformaciones por su propia cuenta.

El objetivo final será que podamos usar nuestro API con un JSON de entrada como el siguiente:

```json
{
  "Pclass": 2,
  "Sex": "male",
  "Age": 46,
  "SibSp": 0,
  "Parch": 0,
  "Fare": 7.2500,
  "Embarked": "C"
}
```

Es decir, **sin transformaciones**.

Lo que haremos es utilizar el objeto `Pipeline` de Scikit-Learn, el cual creará las directrices que permitirán el flujo de datos desde su entrada hasta su salida en forma de predicciones.

## Importar paquetes

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder  # <------------------ Esto es nuevo :)
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.preprocessing import QuantileTransformer
from sklearn.compose import ColumnTransformer    # <------------------ Esto es nuevo :)
from sklearn.model_selection import train_test_split, GridSearchCV

# Pipeline
from sklearn.pipeline import Pipeline           # <------------------ Esto es nuevo :)

# Modelos
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.naive_bayes import GaussianNB, BernoulliNB

# Métricas de evaluación
from sklearn.metrics import accuracy_score

# Para guardar el modelo
import pickle

## Carga de datos

Usaremos los datos limpios del directorio `data/` — **sin** las transformaciones de feature engineering, ya que el pipeline las hará por nosotros.

In [ ]:
df = pd.read_csv('./data/titanic_clean.csv')
df.head()

## Preprocesamiento

A continuación crearemos un objeto especial que usaremos en nuestro pipeline. Este objeto definirá los pasos de preprocesamiento que se ejecutarán tanto en la fase de entrenamiento como en la fase de inferencia.

El `ColumnTransformer` define cuáles y cómo serán las transformaciones que se harán a los datos. Digamos que es nuestro proceso de feature engineering en un solo objeto.

> **¿Qué es `OneHotEncoder`?**  
> En notebooks anteriores usamos `LabelEncoder`. Para propósitos de nuestro proyecto hacen lo mismo. Tenemos que usar `OneHotEncoder` porque `LabelEncoder` no funciona con Pipelines.

In [ ]:
# Definir preprocesamiento
preprocessor = ColumnTransformer(
    transformers=[
        ('onehot', OneHotEncoder(), ['Sex', 'Embarked']),  # OneHotEncoder para columnas categóricas
        ('age',  QuantileTransformer(output_distribution='normal', n_quantiles=500), ['Age']),
        ('fare', QuantileTransformer(output_distribution='normal', n_quantiles=500), ['Fare'])
    ],
    remainder='passthrough'  # Mantener otras columnas sin cambios
)

## Definición de modelos

Nuestro diccionario de modelos tiene un cambio pequeño pero importante: los hiperparámetros llevan ahora el prefijo **`model__`**.

Esto es porque en el Pipeline, el modelo se llama `'model'`. GridSearchCV necesita saber a qué paso del pipeline pertenece cada hiperparámetro, y usa la convención `nombre_paso__hiperparametro`.

In [ ]:
# Definir los modelos y sus respectivos hiperparámetros para GridSearch
modelos = {
    'Regresión Logística': {
        'modelo': LogisticRegression(),
        'parametros': {
            'model__C': [0.01, 0.1, 1, 10, 100],
            'model__penalty': ['l1', 'l2'],
            'model__solver': ['liblinear', 'saga'],
            'model__max_iter': [100, 500, 1000]
        }
    },
    'Clasificador de Vectores de Soporte': {
        'modelo': SVC(),
        'parametros': {
            'model__kernel': ['linear', 'poly', 'rbf', 'sigmoid'],
            'model__C': [0.1, 1, 10]
        }
    },
    'Clasificador de Árbol de Decisión': {
        'modelo': DecisionTreeClassifier(),
        'parametros': {
            'model__splitter': ['best', 'random'],
            'model__max_depth': [None, 1, 2, 3, 4]
        }
    },
    'Clasificador de Bosques Aleatorios': {
        'modelo': RandomForestClassifier(),
        'parametros': {
            'model__n_estimators': [10, 100],
            'model__max_depth': [None, 1, 2, 3, 4],
            'model__max_features': ['sqrt', 'log2', None]
        }
    },
    'Clasificador de Gradient Boosting': {
        'modelo': GradientBoostingClassifier(),
        'parametros': {
            'model__n_estimators': [10, 100],
            'model__max_depth': [None, 1, 2, 3, 4]
        }
    },
    'Clasificador AdaBoost': {
        'modelo': AdaBoostClassifier(),
        'parametros': {
            'model__n_estimators': [10, 100]
        }
    },
    'Clasificador K-Nearest Neighbors': {
        'modelo': KNeighborsClassifier(),
        'parametros': {
            'model__n_neighbors': [3, 5, 7]
        }
    },
    'Clasificador XGBoost': {
        'modelo': XGBClassifier(),
        'parametros': {
            'model__n_estimators': [10, 100],
            'model__max_depth': [None, 1, 2, 3]
        }
    },
    'Clasificador LGBM': {
        'modelo': LGBMClassifier(),
        'parametros': {
            'model__n_estimators': [10, 100],
            'model__max_depth': [None, 1, 2, 3],
            'model__learning_rate': [0.1, 0.2, 0.3],
            'model__verbose': [-1]
        }
    },
    'GaussianNB': {
        'modelo': GaussianNB(),
        'parametros': {}
    },
    'Clasificador Naive Bayes': {
        'modelo': BernoulliNB(),
        'parametros': {
            'model__alpha': [0.1, 1.0, 10.0]
        }
    }
}

## División de datos

Ningún cambio significativo en esta etapa. Nota que ahora `X` conserva las columnas originales con sus valores sin transformar — el pipeline se encargará de procesarlas.

In [ ]:
# División de datos
X = df.drop(['Survived'], axis=1)
y = df['Survived']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=100)

## Variables auxiliares

Ningún cambio significativo en esta etapa.

In [ ]:
# Inicializar variables para almacenar los puntajes de los modelos y el mejor estimador
puntajes_modelos = []
mejor_precision  = 0
mejor_estimador  = None
mejor_modelo     = None
estimadores      = {}

## Ciclo for de GridSearch

El contenido de nuestro ciclo `for` cambia un poco. Principalmente, notamos la creación de un nuevo objeto `Pipeline` **antes** de la creación del objeto `GridSearchCV`:

```python
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('scaler', MinMaxScaler()),
    ('model', info_modelo['modelo'])
])
```

Los pasos (`steps`) que especificamos son:
1. **Preprocesar** los datos con el objeto `preprocessor` (OneHotEncoder + QuantileTransformer)
2. **Escalado** con `MinMaxScaler`
3. **Modelo**

Posteriormente, usamos este `pipeline` como si fuera el estimador en `GridSearchCV`. La diferencia está en que `estimator` ya no es un modelo individual sino un objeto de tipo `Pipeline`.

In [ ]:
# Iterar sobre cada modelo y sus hiperparámetros
for nombre, info_modelo in modelos.items():

    # Crear pipeline para el modelo
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('scaler', MinMaxScaler()),  # MinMaxScaler se aplica a todas las columnas después del preprocesamiento
        ('model', info_modelo['modelo'])  # Modelo placeholder
    ])

    # Inicializar GridSearchCV con el pipeline y los hiperparámetros
    grid_search = GridSearchCV(
        estimator  = pipeline,
        param_grid = info_modelo['parametros'],
        cv         = 5,
        scoring    = 'accuracy',
        verbose    = 0,
        n_jobs     = -1,
    )

    # Ajustar GridSearchCV con los datos de entrenamiento
    grid_search.fit(X_train, y_train)

    # Hacer predicciones con el modelo ajustado
    y_pred = grid_search.predict(X_test)

    # Calcular la precisión de las predicciones
    precision = accuracy_score(y_test, y_pred)

    # Almacenar los resultados del modelo
    puntajes_modelos.append({
        'Modelo':    nombre,
        'Precisión': precision
    })

    estimadores[nombre] = grid_search.best_estimator_

    # Actualizar el mejor modelo si la precisión actual es mayor que la mejor precisión encontrada
    if precision > mejor_precision:
        mejor_modelo    = nombre
        mejor_precision = precision
        mejor_estimador = grid_search.best_estimator_

## Mostrar resultados y guardar modelo

Lo último no sufre ningún cambio. Nota que aunque el código es el mismo, cuando guardamos nuestro pickle **estamos guardando un pipeline**, no un modelo individual.

In [ ]:
# Convertir los resultados a un DataFrame para una mejor visualización
metricas = pd.DataFrame(puntajes_modelos).sort_values('Precisión', ascending=False)

# Imprimir el rendimiento de los modelos de clasificación
print("Rendimiento de los modelos de clasificación")
print(metricas.round(2).to_string())

# Imprimir el mejor modelo y su precisión
print('---------------------------------------------------')
print("MEJOR MODELO DE CLASIFICACIÓN")
print(f"Modelo: {mejor_modelo}")
print(f"Precisión: {mejor_precision:.2f}")

# Guardar el mejor modelo con pickle
with open('pipeline.pkl', 'wb') as archivo_estimador:
    pickle.dump(mejor_estimador, archivo_estimador)

print("\nPipeline guardado exitosamente como 'pipeline.pkl'")

## Warnings

Es muy probable que nuestro GridSearch arroje varios warnings:

```
The max_iter was reached which means the coef_ did not converge
The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6.
```

Es importante leer los warnings, pero **¡los warnings NO son errores!**

Por ejemplo, el primero indica que en alguno de los hiperparámetros probados no fueron adecuados y no pudo converger el proceso de entrenamiento — algo completamente normal cuando GridSearch explora combinaciones subóptimas.